# HR Database — Data Profiling & Regression Analysis

**Tables:** employees, departments, dept_emp, dept_manager, salaries, titles  
**Workflow:**
1. Load CSVs → SQLite `.db`
2. Profile each table
3. Build an analysis-ready flat table
4. Regression: salary ~ tenure, gender, department, title

## 0. Setup

In [ ]:
# !pip install pandas statsmodels matplotlib seaborn

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR   = Path('.')
DB_PATH    = DATA_DIR / 'hr.db'
OUTPUT_DIR = Path('..') / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')
print('Setup complete')
print(f'Outputs → {OUTPUT_DIR.resolve()}')

## 1. Load CSVs → SQLite

In [ ]:
TABLES = ['employees', 'departments', 'dept_emp', 'dept_manager', 'salaries', 'titles']

conn = sqlite3.connect(DB_PATH)

for table in TABLES:
    df = pd.read_csv(DATA_DIR / f'{table}.csv')
    df.to_sql(table, conn, if_exists='replace', index=False)
    print(f'  {table:15s}  {len(df):>10,} rows  {list(df.columns)}')

print(f'\nDatabase saved → {DB_PATH.resolve()}')

## 2. Data Profiling

In [ ]:
profile_frames = {}

def profile_table(name: str) -> pd.DataFrame:
    df = pd.read_sql(f'SELECT * FROM {name}', conn)
    stats = pd.DataFrame({
        'dtype':    df.dtypes,
        'nulls':    df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2),
        'unique':   df.nunique(),
        'sample':   [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    print(f'\n── {name.upper()} ── {len(df):,} rows × {len(df.columns)} cols')
    display(stats)
    profile_frames[name] = stats
    return df

raw = {t: profile_table(t) for t in TABLES}

# Save profiling stats per table
for name, stats in profile_frames.items():
    stats.to_csv(OUTPUT_DIR / f'profile_{name}.csv')
print('\nProfiling CSVs saved to outputs/')

In [ ]:
# Date range check
date_ranges = []
for t in ['employees', 'salaries', 'titles', 'dept_emp']:
    df = raw[t]
    date_cols = [c for c in df.columns if 'date' in c]
    print(f'\n{t}')
    for c in date_cols:
        s = pd.to_datetime(df[c], errors='coerce')
        row = {'table': t, 'column': c, 'min': s.min().date(), 'max': s.max().date(), 'unparseable': s.isna().sum()}
        date_ranges.append(row)
        print(f'  {c}: {row["min"]} → {row["max"]}  ({row["unparseable"]} unparseable)')

pd.DataFrame(date_ranges).to_csv(OUTPUT_DIR / 'profile_date_ranges.csv', index=False)
print('\nDate ranges saved → outputs/profile_date_ranges.csv')

In [ ]:
# Salary distribution
sal = raw['salaries'].copy()
salary_desc = sal['salary'].describe().rename('salary_all_records')
print(salary_desc)
salary_desc.to_csv(OUTPUT_DIR / 'profile_salary_distribution.csv', header=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sal['salary'].plot.hist(bins=60, ax=axes[0], title='Salary Distribution (all records)')
sal['salary'].plot.box(ax=axes[1], title='Salary Box Plot')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_salary_distribution.png', dpi=150)
plt.show()
print('Saved → outputs/chart_salary_distribution.png')

In [ ]:
# Gender split
gender_counts = raw['employees']['gender'].value_counts()
print(gender_counts)
gender_counts.to_csv(OUTPUT_DIR / 'profile_gender_split.csv', header=True)

gender_counts.plot.pie(autopct='%1.1f%%', figsize=(4, 4), title='Gender Split')
plt.ylabel('')
plt.savefig(OUTPUT_DIR / 'chart_gender_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/chart_gender_split.png')

In [ ]:
# Title distribution
title_counts = raw['titles']['title'].value_counts()
print(title_counts)
title_counts.to_csv(OUTPUT_DIR / 'profile_title_counts.csv', header=True)

title_counts.plot.barh(figsize=(7, 4), title='Title Counts (all records incl. history)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_title_counts.png', dpi=150)
plt.show()
print('Saved → outputs/chart_title_counts.png')

## 3. Build Analysis Table

One row per employee using **current** (most recent) salary, title, and department.

In [ ]:
query = """
WITH current_salary AS (
    SELECT emp_no, salary
    FROM salaries
    WHERE to_date = '9999-01-01'
),
current_title AS (
    SELECT emp_no, title
    FROM titles
    WHERE to_date = '9999-01-01'
),
current_dept AS (
    SELECT de.emp_no, de.dept_no, d.dept_name
    FROM dept_emp de
    JOIN departments d ON d.dept_no = de.dept_no
    WHERE de.to_date = '9999-01-01'
)
SELECT
    e.emp_no,
    e.gender,
    e.birth_date,
    e.hire_date,
    cd.dept_name,
    ct.title,
    cs.salary
FROM employees e
JOIN current_salary  cs ON cs.emp_no = e.emp_no
JOIN current_title   ct ON ct.emp_no = e.emp_no
JOIN current_dept    cd ON cd.emp_no = e.emp_no
"""

analysis = pd.read_sql(query, conn)
print(f'{len(analysis):,} active employees loaded')
analysis.head()

In [ ]:
ref_date = pd.Timestamp('2002-08-01')

analysis['birth_date'] = pd.to_datetime(analysis['birth_date'])
analysis['hire_date']  = pd.to_datetime(analysis['hire_date'])

analysis['age_at_hire']   = ((analysis['hire_date'] - analysis['birth_date']).dt.days / 365.25).round(1)
analysis['tenure_years']  = ((ref_date - analysis['hire_date']).dt.days / 365.25).round(2)
analysis['gender_binary'] = (analysis['gender'] == 'M').astype(int)

desc = analysis[['salary', 'tenure_years', 'age_at_hire']].describe()
print(desc)
desc.to_csv(OUTPUT_DIR / 'profile_analysis_flat_summary.csv')
print('Saved → outputs/profile_analysis_flat_summary.csv')

## 4. Exploratory Salary Analysis

In [ ]:
# Avg salary by department
dept_sal = analysis.groupby('dept_name')['salary'].mean().sort_values(ascending=False)
dept_sal.to_csv(OUTPUT_DIR / 'eda_salary_by_department.csv', header=True)

dept_sal.plot.barh(figsize=(8, 5), title='Average Current Salary by Department')
plt.xlabel('Salary ($)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_salary_by_department.png', dpi=150)
plt.show()
print('Saved → outputs/chart_salary_by_department.png')

In [ ]:
# Avg salary by title
title_sal = analysis.groupby('title')['salary'].mean().sort_values(ascending=False)
title_sal.to_csv(OUTPUT_DIR / 'eda_salary_by_title.csv', header=True)

title_sal.plot.barh(figsize=(8, 5), title='Average Current Salary by Title')
plt.xlabel('Salary ($)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_salary_by_title.png', dpi=150)
plt.show()
print('Saved → outputs/chart_salary_by_title.png')

In [ ]:
# Salary by gender
gender_sal = analysis.groupby('gender')['salary'].describe()
print(gender_sal)
gender_sal.to_csv(OUTPUT_DIR / 'eda_salary_by_gender.csv')
print('Saved → outputs/eda_salary_by_gender.csv')

In [ ]:
# Salary vs tenure scatter
fig, ax = plt.subplots(figsize=(9, 5))
for g, grp in analysis.groupby('gender'):
    ax.scatter(grp['tenure_years'], grp['salary'], alpha=0.15, s=5, label=g)
ax.set_xlabel('Tenure (years)')
ax.set_ylabel('Salary ($)')
ax.set_title('Salary vs Tenure by Gender')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_salary_vs_tenure.png', dpi=150)
plt.show()
print('Saved → outputs/chart_salary_vs_tenure.png')

## 5. Regression: Salary ~ Tenure + Gender + Department + Title

In [ ]:
model = smf.ols(
    'salary ~ tenure_years + age_at_hire + gender_binary + C(dept_name) + C(title)',
    data=analysis
).fit()

print(model.summary())

# Save full summary as text
with open(OUTPUT_DIR / 'regression_summary.txt', 'w') as f:
    f.write(model.summary().as_text())
print('Saved → outputs/regression_summary.txt')

In [ ]:
# Coefficients table
coef = pd.DataFrame({
    'coef':    model.params,
    'std_err': model.bse,
    't':       model.tvalues,
    'p_value': model.pvalues,
    'ci_low':  model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).reset_index().rename(columns={'index': 'predictor'})
coef['significant'] = coef['p_value'] < 0.05
coef['abs_coef'] = coef['coef'].abs()
coef = coef.sort_values('abs_coef', ascending=False)

display(coef)
coef.to_csv(OUTPUT_DIR / 'regression_coefficients.csv', index=False)
print('Saved → outputs/regression_coefficients.csv')

In [ ]:
# Residual plots
fitted    = model.fittedvalues
residuals = model.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(fitted, residuals, alpha=0.1, s=4)
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')
pd.Series(residuals).plot.hist(bins=60, ax=axes[1], title='Residual Distribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_regression_residuals.png', dpi=150)
plt.show()

# Model fit metrics
metrics = pd.Series({
    'r_squared':     round(model.rsquared, 4),
    'adj_r_squared': round(model.rsquared_adj, 4),
    'f_statistic':   round(model.fvalue, 2),
    'n_obs':         int(model.nobs),
    'aic':           round(model.aic, 2),
    'bic':           round(model.bic, 2),
}, name='value')
print(metrics)
metrics.to_csv(OUTPUT_DIR / 'regression_model_metrics.csv', header=True)
print('Saved → outputs/chart_regression_residuals.png')
print('Saved → outputs/regression_model_metrics.csv')

## 6. Save Analysis Table Back to SQLite

## 6. Log Transformation Analysis

In [ ]:
# Log transformations of salary
analysis['log_salary']   = np.log(analysis['salary'])    # natural log ln(x)
analysis['log10_salary'] = np.log10(analysis['salary'])  # base-10 log

# Summary comparison
log_summary = pd.DataFrame({
    'raw_salary':   analysis['salary'].describe(),
    'ln_salary':    analysis['log_salary'].describe(),
    'log10_salary': analysis['log10_salary'].describe(),
})
print(log_summary)
log_summary.to_csv(OUTPUT_DIR / 'log_salary_summary.csv')
print('\nSaved → outputs/log_salary_summary.csv')

In [ ]:
# Distribution comparison: raw vs ln vs log10
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

analysis['salary'].plot.hist(bins=60, ax=axes[0], title='Raw Salary', color='steelblue')
axes[0].set_xlabel('Salary ($)')

analysis['log_salary'].plot.hist(bins=60, ax=axes[1], title='ln(Salary) — Natural Log', color='seagreen')
axes[1].set_xlabel('ln(Salary)')

analysis['log10_salary'].plot.hist(bins=60, ax=axes[2], title='log₁₀(Salary) — Base 10', color='tomato')
axes[2].set_xlabel('log₁₀(Salary)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'chart_log_salary_distributions.png', dpi=150)
plt.show()
print('Saved → outputs/chart_log_salary_distributions.png')

In [ ]:
# Re-run OLS using log(salary) as dependent variable and compare R²
model_log = smf.ols(
    'log_salary ~ tenure_years + age_at_hire + gender_binary + C(dept_name) + C(title)',
    data=analysis
).fit()

print(model_log.summary())

# Save log model summary
with open(OUTPUT_DIR / 'regression_log_summary.txt', 'w') as f:
    f.write(model_log.summary().as_text())
print('Saved → outputs/regression_log_summary.txt')

In [ ]:
# Model comparison: raw salary vs log(salary)
comparison = pd.DataFrame({
    'model':       ['OLS (raw salary)', 'OLS (log salary)'],
    'r_squared':   [round(model.rsquared, 4),     round(model_log.rsquared, 4)],
    'adj_r2':      [round(model.rsquared_adj, 4), round(model_log.rsquared_adj, 4)],
    'aic':         [round(model.aic, 2),           round(model_log.aic, 2)],
    'bic':         [round(model.bic, 2),           round(model_log.bic, 2)],
})
print(comparison.to_string(index=False))
comparison.to_csv(OUTPUT_DIR / 'regression_model_comparison.csv', index=False)
print('\nSaved → outputs/regression_model_comparison.csv')

In [ ]:
analysis.to_sql('analysis_flat', conn, if_exists='replace', index=False)
print('Saved analysis_flat table to hr.db')

tables_in_db = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables_in_db)

In [ ]:
conn.close()
print('Connection closed. hr.db is ready for SQL queries.')

print('\nAll outputs saved:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}')